# Цель ноутбука

Сформировать окончательный пул из 20 акций для проекта и проверить, какие данные доступны по каждому инструменту.

Этот ноутбук должен закрыть часть этапов исследования API и фиксации scope: список акций, режим TQBR, глубина дневной и минутной истории, а также ограничения источника.

In [6]:
import pandas as pd
import requests
import time

pd.set_option('display.max_columns', None)

In [2]:
ENGINE = 'stock'
MARKET = 'shares'
BOARD = 'TQBR'

In [3]:
# Получить текущий список инструментов TQBR

url_securities = f'https://iss.moex.com/iss/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities.json'
respons_securities = requests.get(
    url=url_securities,
    timeout=30
)
data_securities = respons_securities.json()
df_securities = pd.DataFrame(
    columns=data_securities['securities']['columns'],
    data=data_securities['securities']['data']

)
secids = df_securities['SECID'].unique().tolist()

In [ ]:
# Получить дневную торговую активность всех бумаг TQBR за 6мес.

params_history = {
    "from": '2026-01-01',
    "till": '2026-07-31',
    "tradingsession": 3,
    "start": 0
}


def get_history(security: str) -> tuple[pd.DataFrame, dict]:
        url= f'https://iss.moex.com/iss/history/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/{security}.json'

        response = requests.get(
            url=url,
            params=params_history,
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        df = pd.DataFrame(
            columns=data['history']['columns'],
            data=data['history']['data'],
        )

        dict_cursor = dict(
            zip(
                data['history.cursor']['columns'],
                data['history.cursor']['data'][0]
                ))
        return df, dict_cursor


data_history = []
for security in secids:
    count_row = 0
    params_history["start"] = 0

    while True:
        df_step, dict_step = get_history(security)

        if df_step.empty:
            if count_row < dict_step["TOTAL"]:
                raise RuntimeError(
                    f"{security}: получено {count_row}"
                    f"из {dict_step['TOTAL']} строк"
                    )
            break

        data_history.append(df_step)
        count_row += df_step.shape[0]

        if count_row < dict_step['TOTAL']:
            params_history["start"] += dict_step['PAGESIZE']
        else:
             break

        time.sleep(0.2)

In [20]:
df_history_securities = pd.concat(data_history, ignore_index=True)

In [8]:
df_history_securities.head()

,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,WAPRICE,CLOSE,VOLUME,MARKETPRICE2,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
0,TQBR,2026-01-05,iАРТГЕН ао,ABIO,1207,6079130.0,68.48,67.64,68.56,67.82,67.9,67.88,89520,67.9,67.9,None,4954777.2,4954777.2,None,0.0,3,SUR,-0.88,2026-01-05
1,TQBR,2026-01-06,iАРТГЕН ао,ABIO,392,2706682.6,67.88,67.78,68.22,68.0,68.0,68.0,39800,68.08,68.08,None,984314.2,984314.2,None,0.0,3,SUR,0.18,2026-01-06
2,TQBR,2026-01-08,iАРТГЕН ао,ABIO,1041,6032664.2,68.0,67.2,68.08,67.66,67.72,67.82,89090,67.74,67.74,None,4423180.0,4423180.0,None,0.0,3,SUR,-0.26,2026-01-08
3,TQBR,2026-01-09,iАРТГЕН ао,ABIO,517,2883348.4,67.72,67.7,68.46,68.14,68.14,67.98,42310,68.16,68.16,None,853246.0,853246.0,None,0.0,3,SUR,0.24,2026-01-09
4,TQBR,2026-01-12,iАРТГЕН ао,ABIO,798,5497470.0,68.0,67.72,68.46,67.92,68.06,67.9,80780,68.06,68.06,None,2867942.2,2867942.2,None,0.0,3,SUR,-0.12,2026-01-12


In [9]:
df_history_securities.tail()


,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,WAPRICE,CLOSE,VOLUME,MARKETPRICE2,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
45134,TQBR,2026-07-27,ЗВЕЗДА ао,ZVEZ,61,629130.0,4.35,4.28,4.47,4.46,4.37,4.47,144000,NaN,4.35,None,0.0,569580.0,None,0.0,3,SUR,3.23,2026-07-27
45135,TQBR,2026-07-28,ЗВЕЗДА ао,ZVEZ,83,716050.0,4.47,4.2,4.5,4.32,4.37,4.5,164000,NaN,4.36,None,0.0,501380.0,None,0.0,3,SUR,0.67,2026-07-28
45136,TQBR,2026-07-29,ЗВЕЗДА ао,ZVEZ,66,503160.0,4.27,4.27,4.61,4.58,4.49,4.61,112000,NaN,4.41,None,0.0,502680.0,None,0.0,3,SUR,2.44,2026-07-29
45137,TQBR,2026-07-30,ЗВЕЗДА ао,ZVEZ,381,5162430.0,4.64,4.63,5.6,4.82,5.01,4.74,1030000,5.08,5.08,None,3309420.0,3309420.0,None,0.0,3,SUR,2.82,2026-07-30
45138,TQBR,2026-07-31,ЗВЕЗДА ао,ZVEZ,32,480230.0,4.84,4.67,4.84,4.74,4.75,4.76,101000,5.08,4.86,None,3361760.0,534700.0,None,0.0,3,SUR,0.42,2026-07-31


In [10]:
df_history_securities.shape

(45139, 24)

In [30]:
mask1 = df_history_securities.groupby("SECID").agg(
    trading_days=("TRADEDATE", "nunique"),
    date_from=("TRADEDATE", "min"),
    date_to=("TRADEDATE", "max"),
).reset_index().sort_values(by='date_from')

mask1 = mask1[mask1['date_from'] == '2026-01-05']
mask1 = mask1[mask1['trading_days'] == 146]

In [42]:
df_activity = df_history_securities[df_history_securities['SECID'].isin(mask1['SECID'].unique())]

df_activity = (
    df_history_securities
    .groupby("SECID", as_index=False)
    .agg(
        median_value=("VALUE", "median"),
        mean_value=("VALUE", "mean"),
        median_numtrades=("NUMTRADES", "median"),
        trading_days=("TRADEDATE", "nunique")
    )
    .sort_values("median_value", ascending=False)
    .reset_index(drop=True)
)

df_activity.index = df_activity.index + 1
df_activity.index.name = "rank"

df_activity.head(25).reset_index()

,rank,SECID,median_value,mean_value,median_numtrades,trading_days
0,1,SBER,7.205961e+09,8.660751e+09,140351.0,146
1,2,AKMM,6.383163e+09,6.709456e+09,19633.0,30
2,3,GAZP,5.893228e+09,7.096368e+09,86774.0,146
3,4,VTBR,4.889119e+09,7.265967e+09,159259.5,146
4,5,T,4.000931e+09,4.516301e+09,60695.0,146
5,6,LKOH,3.935461e+09,4.975364e+09,47928.5,146
6,7,LQDT,3.329981e+09,3.740472e+09,96453.5,30
7,8,NVTK,3.100734e+09,4.140526e+09,65094.5,146
8,9,SBMM,2.976525e+09,3.277067e+09,28017.5,30
9,10,ROSN,2.873151e+09,5.393038e+09,82460.0,146


In [50]:
# Текущий ориентировачный список акций MOEX
selected_securities = [
    "SBER",
    "GAZP",
    "VTBR",
    "T",
    "LKOH",
    "NVTK",
    "ROSN",
    "GMKN",
    "YDEX",
    "PLZL",
    "SMLT",
    "OZON",
    "X5",
    "RUAL",
    "TATN",
    "AFKS",
    "MOEX",
    "PIKK",
    "POSI",
    "AQUA",
]

In [ ]:
# Получить достунпые периоды дат для свечей по интервалу 1 минута
list_securities = []

for security in selected_securities:
    url_candleborders = f'http://iss.moex.com/iss/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/{security}/candleborders.json'
    response = requests.get(
        url = url_candleborders,
        timeout= 30
    )

    data = response.json()
    df = pd.DataFrame(
        columns=data['borders']['columns'],
        data=data['borders']['data']
    )
    df = df[df['interval'] == 1]
    df['SECID'] = security
    list_securities.append(df)


df = pd.concat(list_securities)
df

,begin,end,interval,SECID
0,2011-12-15 10:00:00,2026-08-07 10:59:59,1,SBER
0,2014-06-09 10:00:00,2026-08-07 10:59:59,1,GAZP
0,2013-02-28 10:00:00,2026-08-07 10:59:59,1,VTBR
0,2019-10-28 09:59:00,2026-08-07 10:59:59,1,T
0,2011-12-15 10:00:00,2026-08-07 10:59:59,1,LKOH
0,2011-12-15 10:00:00,2026-08-07 10:59:59,1,NVTK
0,2014-06-09 10:00:00,2026-08-07 10:59:59,1,ROSN
0,2014-06-09 10:00:00,2026-08-07 10:59:59,1,GMKN
0,2024-07-24 09:59:00,2026-08-07 10:59:59,1,YDEX
0,2014-06-09 10:00:00,2026-08-07 10:59:59,1,PLZL
